# LeetCode #1406: Stone Game III

https://leetcode.com/problems/stone-game-iii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(3^n)$ | $O(n)$ |
| **Optimal: Suffix DP ★** | $O(n)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Recursively explore all ways for the current player to take 1, 2, or 3 stones, alternating players without memoization. Exponential branching makes this infeasible for large inputs.

### Optimal: Suffix DP ★
`dp[i]` = maximum score advantage (own score minus opponent's score) the current player can achieve starting from index `i`. Process from right to left: for taking `k` (1, 2, or 3) stones from position `i`, the advantage is `sum(i..i+k) - dp[i+k]`. Take the max over k. Compare `dp[0]` to 0 to determine winner.

**Why this is better than Brute Force:** Memoisation turns the three-way recursion into a single $O(n)$ pass. Rolling 3 variables replaces the full array, giving $O(1)$ space.

**Constraints:**
* $1 \leq$ `stoneValue.length` $\leq 5 \times 10^4$
* $-1000 \leq$ `stoneValue[i]` $\leq 1000$

## Solutions

### C#

In [ ]:
public class Solution {
    public string StoneGameIII(int[] stoneValue) {
        int n = stoneValue.Length;
        // dp[i] = best score advantage the current player can achieve from index i
        // Rolling three variables suffice since we only look ahead 1–3 steps
        int dp1 = 0, dp2 = 0, dp3 = 0; // dp[i+1], dp[i+2], dp[i+3]

        for (int i = n - 1; i >= 0; i--) {
            int best = int.MinValue, sum = 0;
            // Try taking 1, 2, or 3 stones and pick the option maximising advantage
            int[] nexts = { dp1, dp2, dp3 };
            for (int k = 0; k < 3 && i + k < n; k++) {
                sum += stoneValue[i + k];
                // Current player gains sum; opponent plays optimally from i+k+1
                best = Math.Max(best, sum - nexts[k]);
            }
            dp3 = dp2; dp2 = dp1; dp1 = best;
        }

        if (dp1 > 0) return "Alice";
        if (dp1 < 0) return "Bob";
        return "Tie";
    }
}

### Python

In [ ]:
from typing import List

class Solution:
    def stone_game_iii(self, stone_value: List[int]) -> str:
        n = len(stone_value)
        # dp[i] = best score advantage the current player can achieve from index i
        # Rolling three variables suffice since we only look ahead 1-3 steps
        dp = [0] * (n + 3)  # extra padding avoids bounds checks

        for i in range(n - 1, -1, -1):
            best, total = float('-inf'), 0
            for k in range(3):
                if i + k >= n:
                    break
                total += stone_value[i + k]
                # Current player gains total; opponent plays optimally from i+k+1
                best = max(best, total - dp[i + k + 1])
            dp[i] = best

        if dp[0] > 0:
            return "Alice"
        if dp[0] < 0:
            return "Bob"
        return "Tie"

### Go

In [ ]:
func stoneGameIII(stoneValue []int) string {
    n := len(stoneValue)
    // dp[i] = best score advantage the current player can achieve from index i
    dp := make([]int, n+3) // extra padding avoids bounds checks

    for i := n - 1; i >= 0; i-- {
        best, total := -(1 << 30), 0
        for k := 0; k < 3 && i+k < n; k++ {
            total += stoneValue[i+k]
            // Current player gains total; opponent plays optimally from i+k+1
            v := total - dp[i+k+1]
            if v > best { best = v }
        }
        dp[i] = best
    }

    if dp[0] > 0 { return "Alice" }
    if dp[0] < 0 { return "Bob" }
    return "Tie"
}

### Rust

In [ ]:
impl Solution {
    pub fn stone_game_iii(stone_value: Vec<i32>) -> String {
        let n = stone_value.len();
        // dp[i] = best score advantage the current player can achieve from index i
        let mut dp = vec![0i32; n + 3]; // extra padding avoids bounds checks

        for i in (0..n).rev() {
            let mut best = i32::MIN;
            let mut total = 0;
            for k in 0..3 {
                if i + k >= n { break; }
                total += stone_value[i + k];
                // Current player gains total; opponent plays optimally from i+k+1
                best = best.max(total - dp[i + k + 1]);
            }
            dp[i] = best;
        }

        match dp[0].cmp(&0) {
            std::cmp::Ordering::Greater => "Alice".to_string(),
            std::cmp::Ordering::Less    => "Bob".to_string(),
            std::cmp::Ordering::Equal   => "Tie".to_string(),
        }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `stoneValue = [1,2,3,7]`
Bob can ensure 7 by letting Alice take at most 6. dp[0]=-1 → **"Bob"**.

### 2. Slightly Complex
**Input:** `stoneValue = [1,2,3,-9]`
Alice takes 1+2+3=6, leaving -9 for Bob. dp[0]=6-(-9)=15 → **"Alice"**.

### 3. Edge Case: Time Factor
**Input:** `stoneValue` of length $5 \times 10^4$, all values $\pm 1000$.
The DP runs exactly $n$ iterations, each doing 3 comparisons — the full $O(n)$ scan.

### 4. Edge Case: Space Factor
**Input:** `stoneValue` of length $5 \times 10^4$.
With the rolling 3-variable approach, only 3 integers (and a running sum) are maintained — $O(1)$ extra space. The full array `dp` version is $O(n)$.

### 5. Almost-Impossible but Plausible
**Input:** `stoneValue = [0, 0, 0]`
All stones have value 0. Both players score 0 regardless of choices. dp[0]=0 → **"Tie"**.